In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Dashboard Validation Framework — Databricks Entry Point             ║
# ║                                                                       ║
# ║  Schedule this notebook to run BEFORE the dashboard refresh.         ║
# ║  It reads the YAML registry, runs deterministic checks against the   ║
# ║  source tables, and reports PASS / DRIFT / FAIL to Slack.            ║
# ║                                                                       ║
# ║  On FAIL: Claude explains the root cause via the Anthropic API.      ║
# ║  On PASS: a single green line is posted — nothing else.             ║
# ║                                                                       ║
# ║  Parameters (Databricks widgets):                                    ║
# ║    dashboard_name  — key used to locate the registry YAML            ║
# ║    run_week        — fiscal week to validate (YYYY-WW). Leave blank  ║
# ║                      to auto-detect from the current date.          ║
# ║    slack_webhook   — Incoming Webhook URL for the report channel.    ║
# ║                      Leave blank to skip Slack (print only).        ║
# ║    registry_root   — absolute path to the registry folder in the     ║
# ║                      Databricks Repo/Workspace where YAMLs live.    ║
# ╚═══════════════════════════════════════════════════════════════════════╝

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
dbutils.widgets.text("dashboard_name", "social_hub_consolidated_dashboard")
dbutils.widgets.text("run_week",       "")   # blank = auto-detect
dbutils.widgets.text("slack_webhook",  "")   # blank = console only
dbutils.widgets.text("registry_root",  "/Workspace/Repos/mohamed.arshath@latentviewo365.onmicrosoft.com/Dashboard_validation_tool/registry")

DASHBOARD_NAME = dbutils.widgets.get("dashboard_name")
RUN_WEEK_PARAM = dbutils.widgets.get("run_week").strip()
SLACK_WEBHOOK  = dbutils.widgets.get("slack_webhook").strip()
REGISTRY_ROOT  = dbutils.widgets.get("registry_root").strip()

In [ ]:
# ── Resolve run_week ─────────────────────────────────────────────────────────
# If run_week widget is blank, auto-detect by reading MAX(date_column) from
# the dashboard table defined in the YAML registry.

if RUN_WEEK_PARAM:
    RUN_WEEK = RUN_WEEK_PARAM
else:
    import yaml, os
    registry_path = os.path.join(REGISTRY_ROOT, f"{DASHBOARD_NAME}.yaml")
    with open(registry_path, "r") as f:
        reg = yaml.safe_load(f)
    date_col = reg.get("date_column", "fiscal_yr_and_wk_desc")
    dash_tbl = reg["dashboard_table"]
    result = spark.sql(f"SELECT MAX({date_col}) AS latest FROM {dash_tbl}").collect()
    RUN_WEEK = str(result[0]["latest"])

print(f"Dashboard : {DASHBOARD_NAME}")
print(f"Run week  : {RUN_WEEK}")
print(f"Registry  : {REGISTRY_ROOT}")

In [ ]:
# ── Install dependencies if not already present ───────────────────────────────
# pyyaml and anthropic are not pre-installed on all Databricks runtimes.
# Alternatively, add them to the cluster's library list.
%pip install pyyaml "anthropic>=0.40.0" requests "typing_extensions>=4.8.0" --upgrade --quiet

In [ ]:
# ── Add engine + triage to sys.path ──────────────────────────────────────────
import sys, os

# Locate engine/ and triage/ relative to this notebook's location.
# When running from a Databricks Repo, __file__ is set; use REGISTRY_ROOT
# as the anchor to find siblings.
framework_root = str(os.path.dirname(REGISTRY_ROOT))   # dashboard_validation_framework/
engine_dir  = os.path.join(framework_root, "engine")
triage_dir  = os.path.join(framework_root, "triage")

for p in [engine_dir, triage_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"[setup] engine_dir : {engine_dir}")
print(f"[setup] triage_dir : {triage_dir}")

In [ ]:
# ── Run validation ────────────────────────────────────────────────────────────
from validation_engine import ValidationEngine

registry_path = os.path.join(REGISTRY_ROOT, f"{DASHBOARD_NAME}.yaml")
print(f"[validator] Registry path: {registry_path}")

engine = ValidationEngine(spark, registry_path)
run_result = engine.run(run_week=RUN_WEEK)

print(f"[validator] Overall status: {run_result['overall_status']}")

In [ ]:
# ── Console report ────────────────────────────────────────────────────────────
from reporter import print_console_report
print_console_report(run_result)

In [ ]:
# ── Claude triage (FAIL / DRIFT only) ────────────────────────────────────────
from triage_agent import run_triage
from checks import Status

if run_result["overall_status"] != Status.PASS:
    print("[validator] Requesting Claude triage...")
    triage_text = run_triage(run_result)
    run_result["triage_analysis"] = triage_text
    print("[validator] Triage complete")
else:
    run_result["triage_analysis"] = ""

In [ ]:
# ── Slack report ──────────────────────────────────────────────────────────────
from reporter import send_slack_report

if SLACK_WEBHOOK:
    send_slack_report(run_result, SLACK_WEBHOOK)
else:
    print("[validator] No Slack webhook configured — skipping Slack notification")

In [ ]:
# ── Raise on FAIL so the Databricks job turns red ────────────────────────────
# Remove this cell if you want the job to always succeed (e.g. when the
# dashboard refresh should proceed regardless and Slack is the only alert).

if run_result["overall_status"] == Status.FAIL:
    s = run_result["summary"]
    raise RuntimeError(
        f"Dashboard validation FAILED: {s['fail']} failures on {DASHBOARD_NAME} / {RUN_WEEK}. "
        f"Check Slack for details."
    )